> Experiments with [nn.ar](../trapallada/model/ar.py)  module

In [1]:
import jax

import pandas as pd

import trapallada.data
import trapallada.model.ar

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


A sample `Dataset` is loaded

In [3]:
ds = trapallada.data.SignalsDataset(**trapallada.data.sample_dataset_parameters())
len(ds), ds[0][1].shape

(30856, torch.Size([5, 225]))

Inside the `Dataset`, there are `array`s

In [4]:
ds.signals.shape, ds.masks.shape

(torch.Size([30856, 5, 225]), (30856, 225))

The order of the AutoRegressive (AR) process

In [5]:
p = 2
p

2

# statsmodels_window_fit

In [6]:
index = pd.date_range(start="2000-01-01", periods=ds.signals.shape[-1], freq="D")
statsmodel_res = trapallada.model.ar.statsmodels_window_fit(ds.signals[0], index, p)
statsmodel_res

array([-0.06643605,  0.0235036 ,  0.10177639,  0.00812633,  0.07443603,
        0.06527432,  0.00935792,  0.027212  ,  0.03233499, -0.01985912,
        0.04657214,  0.07655839, -0.01704748,  0.00193689,  0.06701202,
       -0.0406343 , -0.02015953,  0.00836119,  0.04086245, -0.01459433,
        0.04464225, -0.01769152,  0.09002308,  0.085877  ,  0.08375366,
       -0.09290047, -0.0514441 , -0.00820403, -0.03816229,  0.01934258,
       -0.01121185,  0.15928221, -0.05923761,  0.10721351, -0.0325327 ,
        0.02767239,  0.02100882, -0.04385914,  0.17554404, -0.03590142,
        0.02543623,  0.05461641,  0.03296202,  0.04072483, -0.07090861,
       -0.04121595,  0.01410057,  0.06193292, -0.06083793, -0.09274973])

# ols_window_fit

In [10]:
jax_res = trapallada.model.ar.ols_window_fit(ds.signals[0].numpy() ,p)
jax_res

Array([-0.066436  ,  0.06527434,  0.04657214, -0.04063427,  0.04464227,
        0.02350358,  0.00935794,  0.07655838, -0.02015957, -0.01769147,
        0.10177642,  0.02721204, -0.01704746,  0.00836118,  0.09002315,
        0.00812634,  0.03233503,  0.00193695,  0.04086242,  0.08587701,
        0.0744361 , -0.0198591 ,  0.06701198, -0.01459433,  0.08375368,
       -0.09290051, -0.01121189,  0.02767239,  0.02543624, -0.04121599,
       -0.05144406,  0.15928227,  0.02100883,  0.0546164 ,  0.01410057,
       -0.00820404, -0.05923762, -0.04385922,  0.03296204,  0.06193296,
       -0.03816233,  0.10721359,  0.17554408,  0.04072484, -0.06083796,
        0.01934261, -0.03253268, -0.03590147, -0.07090859, -0.09274971],      dtype=float32)

Do the two implementation yield the same (*in different order*) coefficients?

In [8]:
np.allclose(np.sort(statsmodel_res), np.sort(jax_res), atol=1e-7)

True

# ols_dataset_fit

In [9]:
trapallada.model.ar.ols_dataset_fit(ds, p).shape

(30856, 50)